# 11.7 新圖文能力進步時，舊文字能力有沒有退步？


手寫成績表裡，圖文題從 2/10 成 7/10，原文字題卻從 8/10 成 6/10。新技能進步可以和舊技能退步同時發生。把兩類並排，才知道學習共享文字權重的代價。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
recipe = {"文字有效位置": 750, "圖文有效位置": 250}
recorded = {
    "訓練前": {"文字答對": 8, "圖文答對": 2},
    "訓練後": {"文字答對": 6, "圖文答對": 7},
}
total_positions = sum(recipe.values())
ratios = {}
for category, positions in recipe.items():
    ratios[category] = positions / total_positions
print("示例訓練比例", ratios)
for phase, scores in recorded.items():
    print(phase, "文字", scores["文字答對"] / 10, "圖文", scores["圖文答對"] / 10)

recipe 將 1,000 個有效位置分成文字 750、圖文 250，印比例 0.75、0.25；recorded 是人工成績示例，兩類各除固定十題，印前後正確率。這不是 75% 回放的訓練結果，而是報告方式。

能力保留是固定原文字題，在訓練前後核對答案。訓練前介面計算一致，不能替代這項檢查。舊文字例可混回訓練，稱回放，但留出題不能混入。

既有小實驗的對照如下，圖文與文字分母不同：

| 更新方案 | 新圖片題 | 原文字題 |
| --- | ---: | ---: |
| 只訓接頭 | 3/12 | 5/10 |
| 接頭＋最後文字區塊 | 12/12 | 2/10 |
| 全部開放 | 9/12 | 1/10 |
| 全部＋文字回放 | 9/12 | 4/10 |

原起點文字都是 5/10。回放版較能保留這批文字題，但沒有完全回到起點；它和其他版本的總有效目標不相同，所以不是嚴格隔離回放比例的因果比較。0.5 抽文字題的機率，也不等於有效位置或時間各半。

實測要看逐題哪個答案改對、哪個改錯，固定生成方式，不只看淨少兩題。混合 loss 下降也可能來自較容易的組佔比變大。練習只改 recipe 為 500／500，比例成各半；recorded 保持原值，因為改計畫不會自行產生成績。

<details>
<summary>回顧與查證</summary>

可回顧：[7.15舊技能遺忘](https://birdhackor.github.io/tiny-perceptron-vlm/7.15.html)、[7.16舊資料回放](https://birdhackor.github.io/tiny-perceptron-vlm/7.16.html)、[10.8純文字入口一致性](https://birdhackor.github.io/tiny-perceptron-vlm/10.8.html)。

原實驗的資料切分、更新設定與逐題結果見[既有實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/vqa.json)；重做入口見[實驗說明](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/README.md)。這是上述有限任務的歷史紀錄。

</details>
